[//]: # (cr:doc name='view_documentation' id=394aee8c)
# View Documentation

This notebook exports the previous notebook's HTML and lists all generated documentation files.

Run this after completing the exploration sequence to ensure all notebooks are exported.

In [ ]:
# @cr:code name='init_progress' id=f56d76fa
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("12_view_documentation.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='check_html_exports' id=4eff77d8
from pathlib import Path

from customer_retention.analysis.notebook_html_exporter import check_exported_html
from customer_retention.core.config.experiments import get_experiments_dir

docs_dir = get_experiments_dir() / "docs"
notebook_dir = Path("exploration_notebooks")

found, missing = check_exported_html(docs_dir, notebook_dir)

print(f"Exported HTML documentation: {len(found)} found, {len(missing)} missing")
print("=" * 60)
for p in found:
    size_kb = p.stat().st_size / 1024
    print(f"  ✓ {p.name:<45} {size_kb:>8.1f} KB")
for stem in missing:
    print(f"  ✗ {stem}")


In [ ]:
# @cr:code name='display_documentation' id=e26dbe37
from customer_retention.analysis.notebook_html_exporter import display_html_documentation

display_html_documentation(docs_dir)


In [ ]:
# @cr:code name='release_stage_memory' id=dbe60cdd
from customer_retention.core.compat import release_stage_memory

release_stage_memory()